# Assignment 14.1 - Generative Adversarial Networks (GAN)

Please submit your solution of this notebook in the Whiteboard at the corresponding Assignment entry as .ipynb-file and as .pdf.

#### Please state both names of your group members here:
S M Shameem Ahmed Khan and Rashid Harvey

## Task 14.1.1: GAN for MNIST

* Implement a basic Generative Adversarial Network (GAN) model and training loop. **(RESULT)**
* Train the GAN on the MNIST dataset. **(RESULT)**
* Visualize some original and generated images from the test set. **(RESULT)**
* Visualize the embeddings of a subset (10 samples) of the test samples using `t-SNE`. You should use your trained encoder model to create those embeddings. **(RESULT)**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# MNIST GAN Implementation

class GAN(nn.Module):
    def __init__(self, learning_rate=0.0002):
        super().__init__()

        self.generator = nn.Sequential(
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 28*28),
            nn.Softmax(dim=1)
        )

        self.discriminator = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

        self.learning_rate = learning_rate
        self.generator_optimizer = optim.Adam(self.generator.parameters(), lr=self.learning_rate)
        self.discriminator_optimizer = optim.Adam(self.discriminator.parameters(), lr=self.learning_rate)

    def train(self, real_data_loader, epochs=50):
        loss_fn = nn.BCELoss()
        for epoch in range(epochs):
            for real_data, _ in real_data_loader:
                batch_size = real_data.size(0)
                real_data = real_data.view(batch_size, -1)

                # Train discriminator
                noise = torch.normal(mean=0.0, std=1.0, size=(batch_size, 256))
                noise = noise + 0.1 * torch.rand(batch_size, 256)
                fake_data = self.generator(noise)

                real_labels = torch.ones(batch_size, 1)
                fake_labels = torch.zeros(batch_size, 1)

                self.discriminator_optimizer.zero_grad()

                # real
                real_output = self.discriminator(real_data)
                d_loss_real = loss_fn(real_output, real_labels)
                d_loss_real.backward()
                
                # fake
                fake_output = self.discriminator(fake_data.detach())
                d_loss_fake = loss_fn(fake_output, fake_labels)
                d_loss_fake.backward()

                self.discriminator_optimizer.step()

                # Train generator
                self.generator_optimizer.zero_grad()
                fake_output = self.discriminator(fake_data)
                g_loss = loss_fn(fake_output, real_labels)
                g_loss.backward()
                self.generator_optimizer.step()

            print(f'Epoch [{epoch+1}/{epochs}], D Loss: {(d_loss_fake+d_loss_real).item()}, G Loss: {g_loss.item()}')

    def generate_samples(self, num_samples=10):
        noise = torch.normal(mean=0.0, std=1.0, size=(num_samples, 256))
        noise = noise + 0.1 * torch.rand(num_samples, 256)
        fake_data = self.generator(noise)
        return fake_data.view(num_samples, 1, 28, 28).detach().numpy()


In [3]:
transform = transforms.ToTensor()
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


gan = GAN(learning_rate=0.0002)
gan.train(train_loader, epochs=50)

Epoch [1/50], D Loss: 8.18304906715639e-05, G Loss: 9.445093154907227
Epoch [2/50], D Loss: 4.290732613299042e-05, G Loss: 10.07102108001709
Epoch [3/50], D Loss: 1.3000321814615745e-05, G Loss: 11.26659870147705
Epoch [4/50], D Loss: 1.8578566596261226e-06, G Loss: 13.20960521697998
Epoch [5/50], D Loss: 3.647969424491748e-05, G Loss: 10.396645545959473
Epoch [6/50], D Loss: 1.0142065548279788e-05, G Loss: 12.199474334716797
Epoch [7/50], D Loss: 6.015880444465438e-06, G Loss: 12.078620910644531
Epoch [8/50], D Loss: 5.661985596816521e-06, G Loss: 12.24538516998291
Epoch [9/50], D Loss: 1.1707545581884915e-06, G Loss: 13.681519508361816
Epoch [10/50], D Loss: 4.0552149016548356e-07, G Loss: 14.748680114746094
Epoch [11/50], D Loss: 4.746954118672875e-07, G Loss: 14.56700325012207
Epoch [12/50], D Loss: 8.951681707003445e-08, G Loss: 16.2724666595459
Epoch [13/50], D Loss: 3.714304241952959e-08, G Loss: 17.109514236450195
Epoch [14/50], D Loss: 5.3354295204144364e-08, G Loss: 16.759393

Something is broken in the training, but I wasn't able to find out what's the problem, even though I tried different network architectures, etc.

## Task 14.1.2: GAN Evaluation

* Evaluate your GAN model on at least 10 generated images, e.g. via Inception Score. **(RESULT)**

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForImageClassification, AutoImageProcessor
from PIL import Image

# We are using a pretrained MNIST classifier from HF
model_name = "farleyknight/mnist-digit-classification-2022-09-04"
image_processor = AutoImageProcessor.from_pretrained(model_name)
classifier = AutoModelForImageClassification.from_pretrained(model_name)
classifier.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = classifier.to(device)

print(f"Loaded pretrained classifier: {model_name}")
print(f"Labels: {classifier.config.id2label}")

def inception_score(gan_model, classifier, image_processor, n_samples=1000, n_splits=10):
    """
    Inception = exp E_x [ D_KL ( p(y|x) || p(y) ) ]
    """
    classifier.eval()

    samples = gan_model.generate_samples(num_samples=n_samples)

    # Convert to PIL
    pil_images = []
    for i in range(n_samples):
        img = (samples[i].squeeze() * 255).astype(np.uint8)
        pil_images.append(Image.fromarray(img, mode="L").convert("RGB"))

    batch_size = 64
    all_probs = []
    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            batch_imgs = pil_images[start : start + batch_size]
            inputs = image_processor(images=batch_imgs, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = classifier(**inputs).logits
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    pyx = np.concatenate(all_probs, axis=0)  # p(y|x), shape (N, 10)

    # Split into groups and compute IS per split
    scores = []
    split_size = n_samples // n_splits
    for k in range(n_splits):
        split = pyx[k * split_size : (k + 1) * split_size]
        py = np.mean(split, axis=0, keepdims=True)
        kl = split * (np.log(split + 1e-16) - np.log(py + 1e-16))
        scores.append(np.exp(np.mean(np.sum(kl, axis=1))))

    # mean and std of IS across splits
    return float(np.mean(scores)), float(np.std(scores))

is_mean, is_std = inception_score(gan, classifier, image_processor, n_samples=1000, n_splits=10)
print(f"\nInception Score: {is_mean:.4f} ± {is_std:.4f}")

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 223.11it/s, Materializing param=vit.layernorm.weight]                                 


Loaded pretrained classifier: farleyknight/mnist-digit-classification-2022-09-04
Labels: {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}

Inception Score: 1.0000 ± 0.0000


## Task 14.1.3: Conditional GAN (cGAN) (BONUS)

* Extend your previous implementations train a GAN including the label information.**(RESULT)**
* Train the cGAN on the MNIST dataset. **(RESULT)**
* Visualize some original and generated images from the test set for each class label. **(RESULT)**

In [ ]:
from IPython.display import Image
Image(url= "https://raw.githubusercontent.com/znxlwm/pytorch-MNIST-CelebA-cGAN-cDCGAN/master/pytorch_cGAN.png", width=900)

In [ ]:
# TODO: Implement

## Congratz, you made it! :)